# MVP architecture anonymizer PII


In [ ]:
# !pip install transformers torch spacy unidecode
# python -m spacy download es_core_news_sm

## 1. Preprocessing layer

In [ ]:
import unicodedata

def strip_accents(text: str) -> str:
    nfc = unicodedata.normalize("NFC", text)
    nfd = unicodedata.normalize("NFD", nfc)
    stripped = "".join(ch for ch in nfd if unicodedata.category(ch) != "Mn")
    return unicodedata.normalize("NFC", stripped)

def preprocess(text: str) -> str:
    # basic normalization
    text = text.strip()
    
    # remove accents
    text_norm = strip_accents(text)
    
    return text, text_norm

## 2. Pattern detection (regex)

In [ ]:
import re

PATTERNS = {
    "EMAIL": r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+",
    "PHONE": r"\+?\d[\d\s\-]{7,}\d",
    "SSN": r"\b\d{10}\b"
}

def extract_regex(text):
    findings = []

    for label, pattern in PATTERNS.items():
        for match in re.finditer(pattern, text):
            findings.append({
                "text": match.group(),
                "entity_group": label,
                "start": match.start(),
                "end": match.end(),
                "source": "regex"
            })

    return findings

## 3. NER detection

OpenMed/OpenMed-PII-Spanish-QwenMed-XLarge-600M-v1

In [ ]:
from transformers import pipeline

ner = pipeline(
    "ner",
    model="OpenMed/OpenMed-PII-Spanish-QwenMed-XLarge-600M-v1",
    aggregation_strategy="simple"
)

def extract_ner(text):
    results = ner(text)

    entities = []
    for r in results:
        entities.append({
            "text": r["word"],
            "entity_group": r["entity_group"],
            "start": r["start"],
            "end": r["end"],
            "score": r["score"],
            "source": "ner"
        })

    return entities

## 4. Entities fusion
Mix regex and NER

In [ ]:
def merge_entities(regex_entities, ner_entities):
    # Sort regex and ner by start
    regex_entities = sorted(regex_entities, key=lambda x: x["start"])
    ner_entities = sorted(ner_entities, key=lambda x: x["start"])

    merged = []

    # 1. Add ALL regex first
    merged.extend(regex_entities)

    # Function to avoid overlap
    def overlaps(ent1, ent2):
        return not (ent1["end"] <= ent2["start"] or ent1["start"] >= ent2["end"])

    # 2. Add NER only if it DOES NOT overlap with regex
    for ner in ner_entities:
        conflict = False
        for reg in regex_entities:
            if overlaps(ner, reg):
                conflict = True
                break
        
        if not conflict:
            merged.append(ner)

    # 3. Final order
    merged = sorted(merged, key=lambda x: x["start"])

    return merged

## 5. PII contextual detection

In [ ]:
def is_pii():
    # Define logic
    return True

## 6. Transformation (anonimization)

In [ ]:
def anonymize(text, entities):
    """Replace detected PII with placeholders."""
    # Sort entities by start position (descending) to preserve offsets
    sorted_entities = sorted(entities, key=lambda x: x['start'], reverse=True)
    redacted = text
    for ent in sorted_entities:
        redacted = redacted[:ent['start']] + f"[{ent['entity_group']}]" + redacted[ent['end']:]
    return redacted

## 7. Complete pipeline

In [ ]:
def process_text(text):

    # 1. Preprocess
    original, norm = preprocess(text)

    # 2. Extract
    regex_entities = extract_regex(norm)
    ner_entities = extract_ner(original)

    # 3. Merge
    entities = merge_entities(regex_entities, ner_entities)

    # 4. Context decision
    for e in entities:
        e["pii"] = is_pii()

    # 5. Anonymize
    output = anonymize(original, entities)

    return {
        "original": original,
        "entities": entities,
        "output": output
    }

## 8. Test

In [ ]:
text = "Me llamo Stiven Moposita, vivo en Quito y mi correo es test@gmail.com mi cédula es 1754650487"

result = process_text(text)

print(result["output"])
print(result["entities"])

In [ ]:
from transformers import AutoModelForTokenClassification, AutoTokenizer
import torch

model_name = "OpenMed/OpenMed-PII-Spanish-QwenMed-XLarge-600M-v1"
model = AutoModelForTokenClassification.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)